# 校准


## 简介

本教程介绍如何使用 **scikit-rf** 校准从 VNA 获取的数据。有关 VNA 校准的入门介绍，请参阅 Rumiantsev 和 Ridler 的文章 [[1](#link_ref1)]，有关不同校准算法的概述，请参阅 Doug Rytting 的演示 [[2](#link_ref2)]。如果您喜欢阅读书籍，可以查看 [[3](#link_ref3)]。

以下内容是如何校准被测器件（DUT）的示例，假设您已经测量了一组可接受的标准件，并拥有一组相应的理想响应。这可称为*离线*校准，因为它不是在 VNA 本身上进行的。这种技术的一个好处是它为非传统校准提供了最大的灵活性，并保留了所有原始数据。

scikit-rf 中实现了许多校准算法。选择哪种算法取决于传输介质、可用的校准标准以及精度和工作量之间的期望权衡。scikit-rf 中可用的校准算法列在 [Calibration](../api/calibration/index.rst) API 页面中。

## 创建校准

校准通过 [Calibration](../api/calibration/index.rst) 类执行。通常，[Calibration](../api/calibration/index.rst) 对象需要两个参数：

* 测量得到的 `Network` 列表
* 理想的 `Network` 列表

每个列表中的 `Network` 元素必须全部相似（相同的端口数、频率信息等），并且必须相互对齐，即理想列表的第一个元素必须对应测量列表的第一个元素。自校准算法（如 TRL）不需要预定义的理想响应。

## 单端口示例

此示例代码旨在提供指导性说明，而非简洁性。要构建单端口校准，您需要至少测量三个标准件，并以 `Network` 形式获得其已知的*理想*响应。这些 `Network` 可以从 touchstone 文件创建，这是所有现代 VNA 都支持的格式。

在以下脚本中，常规短路-开路-负载（SOL）校准套件的测量和理想 touchstone 文件分别位于文件夹 `measured/` 和 `ideal/` 中。这些用于创建 `OnePort` 校准并校正测量的 DUT

	import skrf as rf
	from skrf.calibration import OnePort
	
	## 创建 Calibration 类所需的数据
	
	# Network 类型列表，保存"理想"响应
	my_ideals = [\
	        rf.Network('ideal/short.s1p'),
	        rf.Network('ideal/open.s1p'),
	        rf.Network('ideal/load.s1p'),
	        ]
	
	# Network 类型列表，保存"测量"响应
	my_measured = [\
	        rf.Network('measured/short.s1p'),
	        rf.Network('measured/open.s1p'),
	        rf.Network('measured/load.s1p'),
	        ]
	
	## 创建 Calibration 实例
	cal = OnePort(\
	        ideals = my_ideals,
	        measured = my_measured,
	        )
	
	
	## 运行，并将校准应用于 DUT
	
	# 运行校准算法
	cal.run() 
	
	# 将其应用于 dut
	dut = rf.Network('my_dut.s1p')
	dut_caled = cal.apply_cal(dut)
	dut_caled.name =  dut.name + ' corrected'
	
	# 绘制结果
	dut_caled.plot_s_db()
	# 保存结果 
	dut_caled.write_touchstone()

### 简洁的单端口

此示例以更简洁的*程序化*方式实现与上面相同的任务。

    import skrf as rf
    from skrf.calibration import OnePort
    
    my_ideals = rf.io.load_all_touchstones('ideals/')
    my_measured = rf.io.load_all_touchstones('measured/')
    
    
    ## 创建 Calibration 实例
    cal = OnePort(\
	    ideals = [my_ideals[k] for k in ['short','open','load']],
	    measured = [my_measured[k] for k in ['short','open','load']],
	    )
    
    ## 对 'cal' 的操作可能与上面的示例类似

## 双端口校准

自然地，双端口校准比单端口更复杂。**skrf** 支持几种不同的双端口算法。传统的 `SOLT` 算法使用 12 项误差模型。该算法简单明了，与单端口示例类似。

`EightTerm` 校准基于 R.A.Speciale 在 [[4](#link_ref4)] 中描述的算法。它可以从任意数量的标准件构建，只要满足一些基本约束条件。简而言之，您需要三个双端口标准件；其中一个必须是传输性的，一个必须提供已知阻抗并具有反射性。
请注意，文献中使用的术语*8 项*用于描述各种校准算法使用的特定误差模型，如 TRL、LRM 等。`EightTerm` 类是上述算法的实现，不使用任何自校准。

使用 8 项误差模型公式的一个重要细节是，可能需要进行开关项测量以实现高质量校准（感谢 Dylan Williams 指出这一点）。这些将在下面描述。

### 开关项

最初由 Roger Marks[[5](#link_ref5)] 描述，开关项考虑了 8 项（又称*误差盒*）模型过于简化的事实。两个误差网络会根据哪个端口被激励而略有变化。这是由于 VNA 内部的开关造成的。因此，开关项描述了当源切换到另一个端口时的非理想负载。

开关项可以通过在连接到仪器端口的电缆之间连接低插入损耗传输标准件（如直通）直接测量。有时，VNA 本身上可提供自定义测量配置。

**skrf** 在 `skrf.vi.vna` 模块中支持开关项测量，请参阅 HP8510 的 `skrf.vi.vna.hp.HP8510C.get_switch_terms` 或 PNA 的 `skrf.vi.vna.keysight.get_switch_terms`。如果没有开关项测量，校准质量将取决于 VNA 的特性。

### 在双端口校准中使用单端口理想标准件

通常，您以一端口 touchstone 文件（即 `.s1p`）的形式拥有反射标准件的理想数据。要将其与 skrf 的双端口校准方法一起使用，您需要创建一个由两个网络组成的复合双端口网络。函数 `skrf.network.two_port_reflect` 可以实现这一点
    
    short = rf.Network('ideals/short.s1p')
    shorts = rf.network.two_port_reflect(short, short)

### SOLT 示例

双端口校准与单端口以相同的方式完成，只是所有标准件都是双端口网络。即使对于反射和负载标准件也是如此（S21=S12=0）。因此，如果您测量反射或负载标准件，您应该同时测量两个，并将信息存储在双端口中。例如，将短路连接到端口 1，将短路连接到端口 2，并将双端口测量保存为 'short,short.s2p' 或类似文件。

或者，对于 `SOLT`、`TwelveTerm` 或 `EightTerm` 校准，也可以一次测量一个端口，同时让另一个未使用的端口开路。稍后，可以使用上面解释的 `skrf.network.two_port_reflect` 将两个单端口网络组合成双端口网络。

默认情况下，不进行隔离校准。对于反射甚至负载标准件，$S_{21}$ 和 $S_{12}$ 被忽略。如果需要隔离校准，必须明确告诉 scikit-rf 这样做。具体来说，为此目的使用可选参数 `isolation`。要校准隔离，将此参数设置为双端口网络，表示两个端口都接负载时的测量结果。

应该注意的是，不太常见（但仍然[受支持](../api/calibration/generated/skrf.calibration.calibration.SixteenTerm.rst)）的 `SixteenTerm` 校准是一个显著的例外：该校准专门设计用于比标准 `SOLT` 模型更好地消除端口泄漏。同时测量两个端口的组合标准件是*必需的*。


    import skrf as rf
    from skrf.calibration import SOLT
    
    
    
    # Network 类型列表，保存"理想"响应
    my_ideals = [
	    rf.Network('ideal/short, short.s2p'),
	    rf.Network('ideal/open, open.s2p'),
	    rf.Network('ideal/load, load.s2p'),
	    rf.Network('ideal/thru.s2p'),
        ]
    
    # Network 类型列表，保存"测量"响应
    my_measured = [
	    rf.Network('measured/short, short.s2p'),
	    rf.Network('measured/open, open.s2p'),
	    rf.Network('measured/load, load.s2p'),
	    rf.Network('measured/thru.s2p'),
	    ]
    
    
    ## 创建 SOLT 实例
    cal = SOLT(
	    ideals = my_ideals,
	    measured = my_measured,
        rf.Network('measured/load, load.s2p'),
        # 隔离校准是可选的，可以移除。
	    )
    
    
    ## 运行，并将校准应用于 DUT
    
    # 运行校准算法
    cal.run() 
    
    # 将其应用于 dut
    dut = rf.Network('my_dut.s2p')
    dut_caled = cal.apply_cal(dut)
    
    # 绘制结果
    dut_caled.plot_s_db()
    # 保存结果 
    dut_caled.write_touchstone()

## 保存和调用校准

[Calibration](../api/calibration/index.rst) 可以使用 pickle 的临时存储容器写入和从磁盘读取。写入可以通过使用 `Calibration.write` 或 `rf.io.write()` 完成，读取使用 `rf.io.read()` 完成。由于这些函数依赖于 pickle，因此不推荐用于长期数据存储。目前，除了保存用于生成校准的脚本之外，没有其他方法可以实现校准对象的长期存储。

## 参考文献

<div id='link_ref1'>[1] [Andrej Rumiantsev and Nick Ridler, "VNA Calibration", IEEE Microwave Magazine 2008](http://anlage.umd.edu/Microwave%20Measurements%20for%20Personal%20Web%20Site/Rumiantsev%20VNA%20Calibration%20IEEE%20Mag%20June%202008%20page%2086.pdf)</div> 


<div id='link_ref2'>[2] [Doug Rytting, "Network Analyzer, Error Models and Calibration Methods"](http://emlab.uiuc.edu/ece451/appnotes/Rytting_NAModels.pdf)</div>


<div id='link_ref3'>[3] J. P. Dunsmore, "Handbook of microwave component measurements: with advanced vna techniques". Hoboken, N.J: Wiley, 2012.</div>


<div id='link_ref4'>[4] R.A. Speciale, "A Generalization of the TSD Network-Analyzer Calibration Procedure, Covering n-Port Scattering-Parameter Measurements, Affected by Leakage Errors," Microwave Theory and Techniques, IEEE Transactions on , vol.25, no.12, pp. 1100- 1115, Dec 1977. URL: http://ieeexplore.ieee.org/stamp/stamp.jsp?tp=&arnumber=1129282&isnumber=25047 </div>


<div id='link_ref5'>[5] Roger B. Marks, "Formulations of the Basic Vector Network Analyzer Error Model including Switch-Terms," ARFTG Conference Digest-Fall, 50th , vol.32, no., pp.115-126, Dec. 1997. URL: http://ieeexplore.ieee.org/stamp/stamp.jsp?tp=&arnumber=4119948&isnumber=4119931   </div>